# WCO Forecasting — LSTM Training (Google Colab, exploratory)

> **The canonical training pipeline is now inside the backend**
> (`backend/app/services/training.py`). The API retrains automatically when WCO
> data changes and stores each model version in the database — you do **not**
> need to copy files from this notebook into the backend any more.
> This notebook mirrors the same methodology for exploration and figures.

**Methodology (same as the backend)**
- Each establishment is its own weekly series; 12-week input windows never cross a missing week or another establishment.
- **Chronological split** on calendar weeks: the last 16 weeks are held-out test weeks, the 6 weeks before them are used only for early stopping, everything earlier is training.
- The min–max **scaler is fitted on training-period values only**.
- Seeds are fixed (42).

**Workflow**
1. Export data: `cd backend && ./venv/bin/python -m scripts.export_csv` (columns `establishment_id, wco_code, week_date, quantity_liters, data_source`).
2. Upload `wco_export.csv` here and run all cells.

The model class below **must stay identical** to `build_model` in
`backend/app/services/training.py` (same `SEQUENCE_LENGTH`, `HIDDEN_SIZE`, `NUM_LAYERS`).

## 1. Setup

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# These constants MUST match backend/app/services/training.py
SEQUENCE_LENGTH = 12
HIDDEN_SIZE = 64
NUM_LAYERS = 2
TEST_WEEKS = 16   # held-out evaluation weeks
VAL_WEEKS = 6     # early-stopping weeks before the test period

## 2. Load data

Export from the backend with `python -m scripts.export_csv`, then upload `wco_export.csv` here.
`establishment_id` is only meaningful inside the database it came from — use it for grouping only.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose wco_export.csv
df = pd.read_csv('wco_export.csv', parse_dates=['week_date'])
df = df.sort_values(['establishment_id', 'week_date'])
print(df.shape)
df.head()

## 3. Build training sequences (chronological split, train-only scaler)

Split by calendar date, counted back from the latest week. A window's *target* week decides
whether it is a training, validation or test sample; inputs may reach back into earlier periods
(that is history available at forecast time). Windows never span a gap in the weekly series.

In [ ]:
df['week_date'] = pd.to_datetime(df['week_date'])
last = df['week_date'].max()
test_start = last - pd.Timedelta(weeks=TEST_WEEKS - 1)
val_start = test_start - pd.Timedelta(weeks=VAL_WEEKS)
print('train targets <', val_start.date(), '| validation <', test_start.date(), '| test through', last.date())

# Scaler: TRAINING-PERIOD values only (no leakage from validation/test weeks)
train_vals = df.loc[df['week_date'] < val_start, 'quantity_liters'].to_numpy(dtype=np.float32)
data_min, data_max = float(train_vals.min()), float(train_vals.max())
span = (data_max - data_min) or 1.0
print('scaler min', data_min, 'max', data_max)

parts = {k: ([], []) for k in ('train', 'val', 'test')}
for est_id, g in df.sort_values('week_date').groupby('establishment_id'):
    dates = g['week_date'].to_numpy()
    vals = ((g['quantity_liters'].to_numpy(dtype=np.float32) - data_min) / span)
    # split into runs of exactly-7-day steps
    breaks = np.where(np.diff(dates).astype('timedelta64[D]').astype(int) != 7)[0] + 1
    for run_idx in np.split(np.arange(len(vals)), breaks):
        for i in range(SEQUENCE_LENGTH, len(run_idx)):
            target_date = dates[run_idx[i]]
            key = 'train' if target_date < val_start else ('val' if target_date < test_start else 'test')
            parts[key][0].append(vals[run_idx[i - SEQUENCE_LENGTH:i]])
            parts[key][1].append(vals[run_idx[i]])

def arrays(key):
    X, y = parts[key]
    return np.array(X, dtype=np.float32)[..., None], np.array(y, dtype=np.float32)[..., None]

X_train, y_train = arrays('train')
X_val, y_val = arrays('val')
X_test, y_test = arrays('test')
print('windows  train', len(X_train), ' val', len(X_val), ' test', len(X_test))

g = torch.Generator().manual_seed(SEED)
train_dl = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
                      batch_size=32, shuffle=True, generator=g)
val_dl = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=256)

## 4. Model (keep identical to the backend)

In [ ]:
class WCOLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = WCOLSTM().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)

## 5. Train (with early stopping)

In [ ]:
EPOCHS = 80
PATIENCE = 8
best_val = float('inf')
patience_left = PATIENCE
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        vloss = np.mean([
            criterion(model(xb.to(device)), yb.to(device)).item()
            for xb, yb in val_dl
        ])
    print(f'epoch {epoch:3d}  val_mse {vloss:.5f}')

    if vloss < best_val - 1e-5:
        best_val = vloss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_left = PATIENCE
    else:
        patience_left -= 1
        if patience_left == 0:
            print('Early stopping.')
            break

model.load_state_dict(best_state)

## 6. Evaluate on held-out TEST weeks (liters per establishment-week)

Compare with naive baselines on exactly the same samples. A model is only useful
if it beats them. The backend additionally runs a rolling-origin 13-week recursive
backtest (see `docs/ML_TRAINING_AND_EVALUATION.md`).

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(torch.tensor(X_test).to(device)).cpu().numpy().ravel()
true = y_test.ravel()

pred_l = np.maximum(pred * span + data_min, 0)
true_l = true * span + data_min
naive_l = X_test[:, -1, 0] * span + data_min     # last observed week

def report(name, p):
    mae = mean_absolute_error(true_l, p)
    rmse = mean_squared_error(true_l, p) ** 0.5
    r2 = r2_score(true_l, p)
    mape = np.mean(np.abs((p - true_l) / true_l)[true_l > 0]) * 100
    print(f'{name:22s} MAE {mae:6.3f} L  RMSE {rmse:6.3f} L  R2 {r2:.4f}  MAPE {mape:5.2f}%')

report('LSTM', pred_l)
report('Naive (last week)', naive_l)
print('\nThese are synthetic-data results: they show the pipeline works, not real-world accuracy.')

## 7. (Optional) Export weights + scaler

Not needed for the web app any more — the backend trains and versions its own models.
Kept for offline experiments.

In [ ]:
torch.save(model.state_dict(), 'lstm_wco.pt')
np.savez('scaler.npz', min=data_min, max=data_max)
print('Saved lstm_wco.pt and scaler.npz')

from google.colab import files
files.download('lstm_wco.pt')
files.download('scaler.npz')
print('Place both files in backend/app/services/model_artifacts/')